# CIC-IDS2017: Phase 1 exploration
All computed results describe a bounded, buffered streaming sample. No preprocessing or model training is performed.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import inspect_dataset, load_sample
from src.quality import identify_label, quality_summary, write_report
sns.set_theme(style="whitegrid")
RECORDS = 10000
SEED = 42
BUFFER_SIZE = 10000
SPLIT = "train"

## 1. Inspect metadata and load a sample
Review file sizes, splits, and schema before executing the loading cell. Buffered shuffle can miss rare classes and is not globally uniform. Increase RECORDS deliberately.

In [ ]:
metadata = inspect_dataset()
metadata

In [ ]:
df = load_sample(RECORDS, SEED, BUFFER_SIZE, SPLIT, metadata)
print(f"Examined {len(df):,} rows only")

## 2. First rows

In [ ]:
display(df.head())

## 3. Feature names and types

In [ ]:
display(df.dtypes.rename("dtype").to_frame())
df.info()

## 4. Identify the label column
Match normalized candidate names while preserving original source headers.

In [ ]:
label = identify_label(df)
print(repr(label))
summary = quality_summary(df)

## 5. Count observed attack categories

In [ ]:
counts = df[label].fillna("<MISSING>").value_counts()
display(counts.rename("count").to_frame())

## 6. Sample class imbalance
These proportions cannot estimate full-dataset prevalence reliably. Unobserved attacks may still exist in other files.

In [ ]:
display((100 * counts / len(df)).rename("sample_percent").to_frame())
print("Largest/smallest observed class ratio:", counts.max() / counts.min())

## 7. Missing values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing"))
print("Missing cells:", missing.sum())

## 8. Duplicate records
Count rows beyond their first occurrence using all columns, including label. Do not remove them during exploration.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
display(df[df.duplicated(keep=False)].head())

## 9. Infinite and potentially invalid numerical values
Negative values may be valid sentinels; review their meaning later. Text feature parse failures are reported without modifying the data.

In [ ]:
display(pd.Series(summary["infinite"]).loc[lambda s: s > 0].to_frame("infinite"))
display(pd.Series(summary["nonnumeric_feature_values"], dtype="int64").to_frame("parse_failures"))
display(pd.Series(summary["negative_numeric_values"]).loc[lambda s: s > 0].to_frame("negative"))
print("Constant columns:", summary["constant_columns"])
print("Headers with whitespace:", summary["whitespace_column_names"])

## 10. Visualizations
Log scaling helps reveal rare observed categories. Histogram plotting excludes nonfinite cells for display only.

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(counts) * 0.35)))
sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax)
ax.set(xscale="log", xlabel="Sample records (log scale)", ylabel="Attack category", title="Observed attack distribution")
plt.tight_layout()
plt.show()

In [ ]:
if missing.gt(0).any():
    missing[missing > 0].head(20).plot.barh(title="Missing values in sample", figsize=(9, 5))
    plt.tight_layout()
    plt.show()
else:
    print("No missing values observed in this sample.")
numeric = df.select_dtypes(include="number")
if not numeric.empty:
    column = numeric.columns[0]
    finite = numeric[column].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(finite, bins=40)
    plt.title(f"Sample distribution: {column}")
    plt.tight_layout()
    plt.show()

## 11. Save the data quality report
The same reporting function is available from the CLI. Review these observations before planning Phase 2.

In [ ]:
report_path = write_report(df, metadata, SEED, BUFFER_SIZE, SPLIT)
print(report_path)